# Context Engineering with MCP - Three-App Showcase

**For instructors and learners alike.** A pre-flight tour of the three teaching apps with explanations for the GUI surfaces: MCP Inspector, Claude Code (CC), GitHub Copilot in VS Code.

This notebook is a **hybrid**: runnable cells prove each app works in-process (real tool output, no clicking), and the markdown cells carry the one-paragraph explanation + exact click-path for the three GUI surfaces the notebook cannot click for you.

## The mental model: three apps, a difficulty ramp

| App | Path | Lang | What it is | Primitives |
|-----|------|------|------------|------------|
| **Lab 01 - Hello MCP** | `labs/lab-01-hello-mcp/` | JS | Smallest possible server: one `add` tool | 1 tool |
| **Lab 02 - MCP Chat CLI** | `labs/lab-02-mcp-chat/` | Python | Smallest *complete* system: client + server + REPL | 2 tools, 1 resource + 1 template, 1 prompt |
| **WARNERCO Schematica** | `src/warnerco/backend/` | Python (FastMCP + LangGraph) | Flagship: agentic RAG, 4-tier CoALA memory | 28 tools, 12 resources, 5 prompts |

The pedagogy is a ramp: Lab 01 proves a tool works -> Lab 02 proves a full client/server/REPL loop works -> WARNERCO proves it scales to a real memory-backed agent. **Lab 01 is JS and server-only; Lab 02 is the first one with a client.**

> **Run-from-here note:** the cells below shell out with explicit `cwd`, so you can run this notebook from anywhere. WARNERCO uses the standalone `fastmcp` package; Lab 02 uses the `mcp.server.fastmcp` bundled one. Different import, same `await mcp.get_tools()` introspection API.

## Setup: repo paths

One source of truth for every path used below. Edit `REPO` if your checkout is elsewhere.

In [1]:
import subprocess, sys, json, textwrap
from pathlib import Path

REPO = Path(r"C:\github\context-engineering")
LAB01 = REPO / "labs" / "lab-01-hello-mcp"
LAB02 = REPO / "labs" / "lab-02-mcp-chat"
WARNERCO = REPO / "src" / "warnerco" / "backend"

assert REPO.exists(), f"Repo not found at {REPO} - edit REPO above"
for p in (LAB01, LAB02, WARNERCO):
    print(f"{'OK ' if p.exists() else 'MISSING '} {p}")

OK  C:\github\context-engineering\labs\lab-01-hello-mcp
OK  C:\github\context-engineering\labs\lab-02-mcp-chat
OK  C:\github\context-engineering\src\warnerco\backend


---
# Segment ramp, app 1 of 3: Lab 01 - Hello MCP (JS, server-only)

**The point:** the smallest thing that is a real MCP server. One tool, `add`. No client, no memory, no LangGraph. This proves that an MCP server is just a process that speaks the protocol over stdio.

The cell below does not start a long-running server (that would block the notebook). It confirms the server file is implemented and shows the `add` tool's actual registration and return shape - the human-readable string, **not** `JSON.stringify`.

In [2]:
# Show the actual 'add' tool: name, registration, and TRUE return shape
import re
src = (LAB01 / "solution" / "src" / "index.js").read_text(encoding="utf-8")
name = re.search(r"name:\s*'([^']+)'", src)
print("Server name:", name.group(1) if name else "?")
tool = re.search(r"server\.tool\(\s*'([^']+)'", src)
print("Registered tool:", tool.group(1) if tool else "?")
text_line = re.search(r"text:\s*(`[^`]+`)", src)
print("\nActual returned text:")
print("  " + (text_line.group(1) if text_line else "(not found)"))
print("\nNote: a template string 'The sum of ...', NOT JSON.stringify(result).")

Server name: hello-mcp
Registered tool: add

Actual returned text:
  `The sum of ${a} and ${b} is ${sum}`

Note: a template string 'The sum of ...', NOT JSON.stringify(result).


### GUI surface 1A: showcase Lab 01 in MCP Inspector

Inspector is the **"see the protocol naked"** surface. Run this in a VS Code terminal (it launches a browser UI; do not run it as a notebook cell - it blocks):

```bash
cd labs/lab-01-hello-mcp/starter   # or solution/ for the finished version
npm install
npx @modelcontextprotocol/inspector node src/index.js
```

**Click path:** Inspector opens at `http://localhost:5173` -> **Tools** tab -> you see exactly one tool, `add` -> click it -> enter `a: 5, b: 3` -> **Run** -> response reads `The sum of 5 and 3 is 8`.

**If you're presenting this:** "This is the entire server. One tool. The Inspector is the cleanest way to see what any MCP server exposes before you wire it to a model - tools, resources, prompts, all naked. We start here so that when WARNERCO shows 28 tools, you already know what one tool looks like."

---
# App 2 of 3: Lab 02 - MCP Chat CLI (Python, first one with a CLIENT)

**The point:** the smallest *complete* system. A Python MCP **client**, a stdio FastMCP **server**, and a chat **REPL**, all in one tiny lab. This is the conceptual bridge: Lab 01 had no client, WARNERCO has too much to read in one sitting. Lab 02 is the Goldilocks middle.

The cell below introspects the Lab 02 server **live** - actual decorator counts, no hardcoded numbers.

In [3]:
# Introspect Lab 02 server in-process via its own venv (uv run resolves it).
# NOTE the API difference: Lab 02 uses the BUNDLED mcp.server.fastmcp.FastMCP,
# whose introspection methods are list_tools()/list_resources()/list_prompts()
# (low-level MCP names). WARNERCO uses the STANDALONE fastmcp package, whose
# methods are get_tools()/get_resources()/get_prompts(). Same protocol, two SDKs.
probe = r'''
import asyncio
from mcp_server import mcp
async def go():
    t  = await mcp.list_tools()
    r  = await mcp.list_resources()
    rt = await mcp.list_resource_templates()
    p  = await mcp.list_prompts()
    print("tools:    ", [x.name for x in t])
    print("resources:", [str(x.uri) for x in r])
    print("templates:", [x.uriTemplate for x in rt])
    print("prompts:  ", [x.name for x in p])
asyncio.run(go())
'''
r = subprocess.run(["uv", "run", "python", "-c", probe], cwd=LAB02,
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

tools:     ['read_doc_contents', 'edit_document']
resources: ['docs://documents']
templates: ['docs://documents/{doc_id}']
prompts:   ['format']



You should see **2 tools** (`read_doc_contents`, `edit_document`), **1 static resource + 1 template** (`docs://documents`, `docs://documents/{doc_id}`), **1 prompt** (`format`). The only working slash command in the REPL is `/format` (there is no `/summarize` - that was an upstream TODO never implemented).

### GUI surface 2A: showcase Lab 02 in Claude Code

Lab 02 is best shown as a **live chat loop** - its whole reason to exist is the client+REPL. Run its on-rails launcher in a VS Code terminal:

```powershell
cd labs/lab-02-mcp-chat
.\run.ps1
```

`run.ps1` bootstraps the lab `.env`, lifts `ANTHROPIC_API_KEY` from the repo-root `.env` (so you do not paste the key twice), and hands off to the REPL.

**If you're presenting this:** "Lab 01 was a server with no brain. Now we add the two missing pieces - a client that speaks MCP, and a model behind it. Type `/format <doc>` and watch the prompt primitive fire. This is the full loop in about 200 lines, before we scale it up."

---
# App 3 of 3: WARNERCO Schematica (the flagship)

**The point:** everything at production scale. FastMCP + LangGraph 9-node RAG pipeline + four CoALA memory tiers (Working / Episodic / Semantic / Procedural). This is the star of all three GUI surfaces.

The cell below introspects WARNERCO **live** and prints the real primitive counts. **Do not hardcode these on a slide** - they drift fastest in a teaching repo, which is exactly why this cell exists.

In [4]:
# Live primitive census - the number to trust comes from HERE (run it live), not memory
probe = '''
import asyncio
from app.mcp_tools import mcp
async def go():
    tools = await mcp.get_tools()
    res = await mcp.get_resources()
    tmpl = await mcp.get_resource_templates()
    prompts = await mcp.get_prompts()
    print(f"TOOLS:     {len(tools)}")
    print(f"RESOURCES: {len(res)} static + {len(tmpl)} template = {len(res)+len(tmpl)} total")
    print(f"PROMPTS:   {len(prompts)}")
    print()
    print("Template (the +1):", list(tmpl))
    print()
    print("All tool names:")
    for i, t in enumerate(sorted(tools), 1):
        print(f"  {i:2d}. {t}")
asyncio.run(go())
'''
r = subprocess.run(["uv", "run", "python", "-c", probe], cwd=WARNERCO,
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

TOOLS:     28
RESOURCES: 11 static + 1 template = 12 total
PROMPTS:   5

Template (the +1): ['schematic://{schematic_id}']

All tool names:
   1. warn_add_relationship
   2. warn_compare_schematics
   3. warn_consolidate_memory
   4. warn_create_schematic
   5. warn_delete_schematic
   6. warn_describe_tool
   7. warn_episodic_log
   8. warn_episodic_recall
   9. warn_episodic_recent
  10. warn_episodic_stats
  11. warn_explain_schematic
  12. warn_feedback_loop
  13. warn_get_robot
  14. warn_graph_neighbors
  15. warn_graph_path
  16. warn_graph_stats
  17. warn_guided_search
  18. warn_index_schematic
  19. warn_list_robots
  20. warn_memory_stats
  21. warn_replacement_advisor
  22. warn_scratchpad_clear
  23. warn_scratchpad_read
  24. warn_scratchpad_stats
  25. warn_scratchpad_write
  26. warn_search_tools
  27. warn_semantic_search
  28. warn_update_schematic



**Teaching micro-point on the count:** the runtime API reports static resources and templates separately. `get_resources()` returns 11, `get_resource_templates()` returns 1 (`schematic://{schematic_id}`), summing to the **12** total. When a client hides templates and shows only 11, that is the gap - not drift.

In [5]:
# Call a real tool in-process and show actual output (no Inspector needed)
probe = r'''
import asyncio
from app.mcp_tools import mcp
async def go():
    tools = await mcp.get_tools()
    res = await tools["warn_search_tools"].fn(query="", detail="name")
    print(f'warn_search_tools(query="", detail="name") -> count={res.count} total={res.total}')
    print("  (default limit=20 caps count at 20; total is the real 28)")
    narrow = await tools["warn_search_tools"].fn(query="graph", detail="summary")
    print(f'\nwarn_search_tools(query="graph") -> count={narrow.count} tools matched')
    for t in narrow.tools:
        print(f"  {t['name']}: {t['summary']}")
asyncio.run(go())
'''
r = subprocess.run(["uv", "run", "python", "-c", probe], cwd=WARNERCO,
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

warn_search_tools(query="", detail="name") -> count=20 total=28
  (default limit=20 caps count at 20; total is the real 28)

warn_search_tools(query="graph") -> count=8 tools matched
  warn_add_relationship: Add a relationship between two entities in the knowledge graph.
  warn_create_schematic: Create a new robot schematic in the database.
  warn_explain_schematic: Get an LLM-enriched explanation of a robot schematic.
  warn_graph_neighbors: Get all entities connected to the given entity in the knowledge graph.
  warn_graph_path: Find the shortest path between two entities in the knowledge graph.
  warn_graph_stats: Get statistics about the knowledge graph.
  warn_replacement_advisor: Interactive component replacement wizard using multi-step elicitation.
  warn_semantic_search: Search robot schematics using natural language queries.



### GUI surface 3A: WARNERCO in MCP Inspector

```bash
cd src/warnerco/backend
npx @modelcontextprotocol/inspector uv run warnerco-mcp
```

**Click path:** `http://localhost:5173` -> **Tools** tab shows all 28 -> run `warn_search_tools` with `{"query":"","detail":"name"}` to demo progressive loading -> **Resources** tab -> open `memory://coala-overview` for the live four-tier snapshot -> **Prompts** tab shows the 5.

**If you're presenting this:** "Same Inspector as Lab 01, wildly different server. Twenty-eight tools. Rather than dump every schema at the model, WARNERCO ships a discovery tool - `warn_search_tools` - so the model loads only what it needs. That is the 'code execution with MCP' pattern Anthropic published."

### GUI surface 3B: WARNERCO in Claude Code

Already wired in `.claude/mcp.json` - **two entries on purpose**:
- `claude-warnerco-schematica` - general entry (`uv run warnerco-mcp`, `MEMORY_BACKEND=chroma`)
- `claude-warnerco-coala-memory` - same binary, pre-pins the `EPISODIC_*` weights (0.4 / 0.3 / 0.3, half-life 24h) so the memory demo is class-stable

There is no separate CoALA process; the second entry is pedagogical clarity. **If you're presenting this:** "Watch Claude reason across the four memory tiers. I pinned the episodic weights in the second config so the recall scores are identical every run - no surprises live."

### GUI surface 3C: WARNERCO in GitHub Copilot (VS Code, MSFT extension)

`.vscode/mcp.json` defines **two servers** (note: VS Code uses the `servers` root key and requires `type`):
- `vscode-warnerco-schematica` - the app (`type: stdio`, `uv run warnerco-mcp`, `MEMORY_BACKEND=chroma`)
- `ghcopilot-warner` - GitHub's remote MCP server (`type: http`, `https://api.githubcopilot.com/mcp/`, PAT via `${input:github_pat}`)

**Click path:** open the repo in VS Code -> Copilot Chat -> switch to **Agent** mode -> the MCP tools picker lists both servers -> first run prompts for your GitHub PAT (the `github_pat` input) -> ask Copilot something that hits a `warn_*` tool. **If you're presenting this:** "Here is the enterprise story: my MCP server and GitHub's, side by side in the same Copilot agent. The IDE does not care whose server it is - it is all the same protocol."

## Pre-flight: confirm all three GUI surfaces will launch

Run this last cell before you present (or before you start the labs). It does **not** open the GUIs (they block); it confirms the launch prerequisites exist so nothing surprises you when you go live.

In [6]:
checks = []
# uv present
uv = subprocess.run(["uv", "--version"], capture_output=True, text=True)
checks.append(("uv on PATH", uv.returncode == 0, uv.stdout.strip()))
# npx present (for Inspector)
npx = subprocess.run(["npx", "--version"], capture_output=True, text=True, shell=True)
checks.append(("npx on PATH (Inspector)", npx.returncode == 0, npx.stdout.strip()))
# repo-root .env has the key (Lab 02 + WARNERCO need it)
env = (REPO / ".env")
has_key = env.exists() and "ANTHROPIC_API_KEY=sk-ant-" in env.read_text(encoding="utf-8")
checks.append(("repo-root .env has live-shaped key", has_key, str(env)))
# both MCP client configs present
checks.append((".claude/mcp.json present", (REPO/".claude/mcp.json").exists(), ""))
checks.append((".vscode/mcp.json present", (REPO/".vscode/mcp.json").exists(), ""))

print("PRE-FLIGHT\n" + "="*40)
for label, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}  {detail}")
if all(ok for _, ok, _ in checks):
    print("\nAll surfaces clear to launch.")
else:
    print("\nFix the FAIL rows before class.")

PRE-FLIGHT
  [PASS] uv on PATH  uv 0.9.26 (ee4f00362 2026-01-15)
  [PASS] npx on PATH (Inspector)  11.7.0
  [PASS] repo-root .env has live-shaped key  C:\github\context-engineering\.env
  [PASS] .claude/mcp.json present  
  [PASS] .vscode/mcp.json present  

All surfaces clear to launch.
